# SqueakView Short-run Analysis Example

This teaching notebook demonstrates protocol-v2 camera-anchor reconstruction, durable controller events, detections, NvDCF tracks, and pose keypoints from the current DeepStream 9.1 schema. Run it on the analysis device with a short or sampled copied run; it loads canonical tables into memory and is not the production-scale workflow for a full long-duration recording. The copied `raw.mp4`, `frames.csv`, and `diagnostics/controller_v2.jsonl` remain ground truth. See the adjacent [workflow guide](README.md) for the transfer, validation, and source-data rules.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Rectangle

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "pyproject.toml").exists()), HERE)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data_viz.v2_alignment import (
    CAMERA_RECORDS,
    associate_events_to_frames,
    find_latest_run,
    load_v2_run,
)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Leave RUN_DIR=None to use the newest project run, or set a path explicitly.
RUN_DIR = None
# "live", "latest_offline", or a Path to one offline_inference result directory.
INFERENCE_RESULT = "live"

if RUN_DIR is None:
    RUN_DIR = find_latest_run()
RUN_DIR = Path(RUN_DIR).expanduser().resolve()

if INFERENCE_RESULT == "live":
    RESULT_DIR = RUN_DIR
elif INFERENCE_RESULT == "latest_offline":
    candidates = sorted((RUN_DIR / "offline_inference").glob("*"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("No offline inference result exists for this run")
    RESULT_DIR = candidates[-1]
else:
    RESULT_DIR = Path(INFERENCE_RESULT).expanduser().resolve()
print(f"RUN_DIR:      {RUN_DIR}")
print(f"RESULT_DIR:   {RESULT_DIR}")


## Load Current Tables

The durable protocol-v2 journal supplies exact `CAMERA_EPOCH`, periodic `CAMERA_CHECKPOINT`, and final `CAMERA_STOP` anchors. The helper validates the run and reconstructs intervening frame timestamps with an explicit `piecewise_interpolated` label. Object, keypoint, and track tables come from the selected inference result.


In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def numeric(df: pd.DataFrame, columns) -> pd.DataFrame:
    for column in columns:
        if column in df:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df

v2_run = load_v2_run(RUN_DIR)
frames = v2_run.frames.copy()
events = v2_run.events.copy()
anchors = v2_run.anchors.copy()
aligned_events = associate_events_to_frames(events, frames)
objects = numeric(read_csv(RESULT_DIR / "objects.csv"), [
    "deepstream_frame_number", "source_sequence_index", "camera_frame_id", "camera_timestamp_ns",
    "gst_pts_ns", "class_id", "track_id", "detected_this_frame", "tracker_predicted",
    "detector_confidence", "tracker_confidence", "track_x", "track_y", "track_w", "track_h",
])
keypoints = numeric(read_csv(RESULT_DIR / "keypoints.csv"), [
    "deepstream_frame_number", "source_sequence_index", "camera_frame_id", "track_id", "class_id",
    "keypoint_index", "x_px", "y_px", "x_norm", "y_norm", "confidence", "visible",
])

frames["frame_pts_ns"] = frames["pts_ns"].fillna(frames.get("gst_pts_ns"))
frames["frame_pts_s"] = frames["frame_pts_ns"] / 1_000_000_000
object_counts = objects.groupby("source_sequence_index").size() if not objects.empty else pd.Series(dtype=int)
frames["detection_count"] = frames["source_sequence_index"].map(object_counts).fillna(0).astype(int)
frames["has_detection"] = (frames["detection_count"] > 0).astype(int)

frame_lookup = frames[["source_sequence_index", "raw_frame_index", "camera_frame_id", "controller_count", "frame_controller_us", "controller_time_method", "frame_time_s", "frame_pts_ns", "frame_pts_s"]].drop_duplicates("source_sequence_index")
if not objects.empty:
    objects = objects.merge(frame_lookup, on="source_sequence_index", how="left", suffixes=("", "_ledger"), validate="many_to_one")
if not keypoints.empty:
    keypoints = keypoints.merge(frame_lookup, on="source_sequence_index", how="left", suffixes=("", "_ledger"), validate="many_to_one")

detections = objects.copy()
if not detections.empty:
    detections["detection_index"] = np.arange(len(detections))
    detections["detection_controller_us"] = detections["frame_controller_us"]
    detections["detection_time_s"] = detections["frame_time_s"]
    detections["raw_frame_mapping_method"] = "offline_video_ledger" if RESULT_DIR != RUN_DIR else "flir_user_meta"
    detections["raw_frame_mapping_ok"] = detections["camera_frame_id_ledger"].notna().astype(int) if "camera_frame_id_ledger" in detections else detections["camera_frame_id"].notna().astype(int)
    detections["raw_frame_mapping_pts_ns"] = detections["gst_pts_ns"]
    detections["conf"] = detections["detector_confidence"]
    detections["x"], detections["y"] = detections["track_x"], detections["track_y"]
    detections["w"], detections["h"] = detections["track_w"], detections["track_h"]
    detections["original_frame"] = detections["deepstream_frame_number"]

tracked = objects[objects["track_id"].notna() & (objects["track_id"] >= 0)].copy() if not objects.empty else pd.DataFrame()
if tracked.empty:
    tracks = pd.DataFrame()
else:
    tracks = tracked.groupby(["stream_id", "track_id", "class_id", "class_label"], as_index=False).agg(
        first_frame=("source_sequence_index", "min"), last_frame=("source_sequence_index", "max"),
        observed_frames=("detected_this_frame", "sum"), predicted_frames=("tracker_predicted", "sum"),
        total_rows=("observation_id", "size"),
    )

print(f"frames={len(frames):,} anchors={len(anchors):,} events={len(events):,} detections={len(detections):,}")
print(f"objects={len(objects):,} keypoints={len(keypoints):,} tracks={len(tracks):,}")
display(frames.head(3), objects.head(3), keypoints.head(3), tracks.head(3))


## Acquisition and Protocol-v2 Health

Inspect this before interpreting behavior. The loader has already rejected a non-finalized run, a failed recording audit, transport corruption, a boot boundary, malformed camera anchors, or a controller/frame-count mismatch. `alignment_validated=false` is expected because v2 does not use the legacy per-edge aligner.


In [ ]:
display(pd.Series(v2_run.transport_summary.get("counts", {}), name="v2 transport").to_frame())
display(pd.DataFrame(v2_run.status["recording_validation"]["cameras"]))
display(pd.DataFrame(v2_run.status["acquisition_integrity"]["cameras"]))
display(anchors)
display(frames["controller_time_method"].value_counts().to_frame("frames"))

errors = read_csv(RUN_DIR / "diagnostics" / "errors.csv")
if not errors.empty:
    display(errors)

if RESULT_DIR != RUN_DIR:
    offline_manifest = json.loads((RESULT_DIR / "offline_manifest.json").read_text())
    display(pd.Series({
        "status": offline_manifest.get("status"),
        "decoded_frames": offline_manifest.get("decoded_frames"),
        "expected_frames": offline_manifest.get("source", {}).get("expected_frames"),
        "model": offline_manifest.get("model_package", {}).get("name"),
        "video_sha256": offline_manifest.get("source", {}).get("video_sha256"),
    }, name="offline provenance").to_frame())
else:
    admission_path = RUN_DIR / "inference_admission.json"
    if admission_path.exists():
        display(pd.Series(json.loads(admission_path.read_text()), name="live admission").to_frame())


## Frame Timing

The camera timestamp and GStreamer PTS are exact per-frame clocks. Protocol-v2 controller timestamps are exact at `EPOCH`, `CHECKPOINT`, and `STOP` anchors and piecewise-interpolated between them; they must not be presented as measured per-frame TTL edges.


In [ ]:
frames = frames.sort_values("source_sequence_index").reset_index(drop=True)
frames["controller_interval_ms"] = frames["frame_controller_us"].diff() / 1_000.0
frames["camera_interval_ms"] = frames["camera_timestamp_ns"].diff() / 1_000_000.0
frames["pts_interval_ms"] = frames["frame_pts_ns"].diff() / 1_000_000.0
run_manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
fps = float(run_manifest.get("capture", {}).get("fps", 30))
expected_ms = 1_000.0 / fps

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(frames["camera_frame_id"], frames["camera_interval_ms"], ".-", ms=3, lw=.8, label="FLIR camera clock")
axes[0].plot(frames["camera_frame_id"], frames["pts_interval_ms"], ".-", ms=3, lw=.8, label="GStreamer PTS")
axes[0].plot(frames["camera_frame_id"], frames["controller_interval_ms"], lw=.8, alpha=.7, label="v2 reconstructed controller clock")
axes[0].axhline(expected_ms, color="black", ls="--", lw=1, label=f"{fps:g} fps target")
axes[0].set(xlabel="FLIR camera_frame_id", ylabel="interval (ms)", title="Frame-to-frame interval")
axes[0].legend()
sns.histplot(frames["camera_interval_ms"].dropna(), bins=40, ax=axes[1], label="FLIR camera", color="tab:blue")
sns.histplot(frames["pts_interval_ms"].dropna(), bins=40, ax=axes[1], label="PTS", color="tab:orange", alpha=.5)
axes[1].axvline(expected_ms, color="black", ls="--", lw=1)
axes[1].set(xlabel="interval (ms)", title="Interval distributions")
axes[1].legend()
plt.tight_layout()
display(frames[["camera_interval_ms", "pts_interval_ms", "controller_interval_ms"]].describe(percentiles=[.01, .05, .5, .95, .99]))


## Inference Coverage and Serial Events

Detection coverage is aligned to every recorded frame. Tracker-predicted object rows are intentionally distinguished from detector observations.


In [ ]:
non_camera = events[~events["eventType"].isin(CAMERA_RECORDS)].dropna(subset=["event_time_s"])
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(frames["frame_time_s"], frames["detection_count"], lw=1.2, label="detection rows/frame")
if not detections.empty:
    ax.scatter(detections["detection_time_s"], detections["conf"], s=10, alpha=.4, label="detector confidence")
for _, row in non_camera.iterrows():
    ax.axvline(row["event_time_s"], color="tab:red", alpha=.15, lw=.8)
ax.set(xlabel="time from CAMERA_EPOCH (s)", ylabel="count / confidence", title="Inference coverage on reconstructed controller time")
ax.legend()
plt.tight_layout()

if not objects.empty:
    display(objects[["detected_this_frame", "tracker_predicted"]].sum().rename({
        "detected_this_frame": "detector observations", "tracker_predicted": "tracker-only predictions"
    }).to_frame("rows"))


## Serial Event Raster


In [ ]:
plot_events = events[~events["eventType"].isin(CAMERA_RECORDS)].dropna(subset=["event_time_s"]).copy()
if plot_events.empty:
    print("No non-camera serial events found.")
else:
    fig, ax = plt.subplots(figsize=(14, max(3, .35 * plot_events["eventType"].nunique() + 2)))
    sns.scatterplot(
        data=plot_events,
        x="event_time_s",
        y="eventType",
        hue="side" if "side" in plot_events else None,
        s=65,
        ax=ax,
    )

    pellet_events = plot_events[plot_events["eventType"].str.contains("PELLET|WELL_CHECK", na=False)].copy()
    if not pellet_events.empty:
        palette = {
            "PELLET_ARRIVAL": "tab:orange",
            "PELLET_RETRIEVAL": "tab:red",
            "WELL_CHECK_START": "tab:green",
            "WELL_CHECK_END": "tab:olive",
        }
        for event_type, color in palette.items():
            subset = pellet_events[pellet_events["eventType"] == event_type]
            if subset.empty:
                continue
            ax.scatter(
                subset["event_time_s"],
                subset["eventType"],
                color=color,
                marker="X" if "WELL_CHECK" in event_type else "D",
                s=120,
                edgecolors="black",
                linewidths=1.0,
                label=event_type,
                alpha=0.9,
            )

    ax.set(xlabel="time from CAMERA_EPOCH (s)", ylabel="event type", title="Durable protocol-v2 controller events")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.0)
    plt.tight_layout()
    display(plot_events["eventType"].value_counts().to_frame("count"))


In [ ]:
def infer_pellet_serial_mode(events: pd.DataFrame) -> str:
    pellet_events = events[events["eventType"].str.contains("PELLET|WELL_CHECK", na=False)]
    if pellet_events.empty:
        return "unknown"
    has_arrival = pellet_events["eventType"].str.contains("ARRIVAL|START", na=False).any()
    has_retrieval = pellet_events["eventType"].str.contains("RETRIEVAL|END", na=False).any()
    if has_arrival and has_retrieval:
        return "both"
    if has_arrival:
        return "arrival"
    if has_retrieval:
        return "retrieval"
    return "unknown"

pellet_mode = infer_pellet_serial_mode(events)
print(f"Inferred pellet telemetry mode: {pellet_mode}")

pellet_events = events[events["eventType"].str.contains("PELLET|WELL_CHECK", na=False)].dropna(subset=["event_time_s"]).copy()
if pellet_events.empty:
    print("No PELLET or WELL_CHECK events found in the v2 journal.")
else:
    pellet_events["short_event"] = (
        pellet_events["eventType"]
        .str.replace("PELLET_", "", regex=False)
        .str.replace("WELL_CHECK_", "WELL_", regex=False)
    )
    pellet_events["group"] = np.where(
        pellet_events["eventType"].str.contains("WELL_CHECK", na=False),
        "WELL_CHECK",
        "PELLET",
    )
    fig, ax = plt.subplots(figsize=(14, 4))
    palette = {"PELLET": "tab:orange", "WELL_CHECK": "tab:green"}
    for group, group_df in pellet_events.groupby("group"):
        ax.scatter(group_df["event_time_s"], group_df["short_event"], color=palette[group], s=80, alpha=.8, label=group)
    ax.set(
        xlabel="time from CAMERA_EPOCH (s)",
        ylabel="pellet / well event",
        title="Pellet and well-check serial event timeline",
    )
    ax.legend()
    plt.tight_layout()
    display(pellet_events[["event_time_s", "eventType", "side", "count"]])

## Detection-to-Frame Mapping Proof

Live outputs should map by `flir_user_meta`; offline outputs should map by `offline_video_ledger`. Both preserve `source_sequence_index` and join exactly to the validated frame ledger. Controller count and reconstructed controller time then come from the protocol-v2 camera anchors.


In [ ]:
if detections.empty:
    print("No detections to audit.")
else:
    method_counts = detections["raw_frame_mapping_method"].fillna("missing").value_counts()
    display(method_counts.to_frame("rows"))
    map_check = detections.merge(
        frames[["camera_frame_id", "frame_pts_ns"]], on="camera_frame_id", how="left",
        suffixes=("_det", "_ledger"),
    )
    map_check["mapping_pts_delta_ns"] = map_check["raw_frame_mapping_pts_ns"] - map_check["frame_pts_ns_ledger"]
    problems = map_check[(map_check["raw_frame_mapping_ok"].fillna(0) != 1) | map_check["camera_frame_id"].isna()]
    print(f"problem mapping rows: {len(problems):,}")
    display(map_check[["detection_index", "raw_frame_index", "camera_frame_id", "raw_frame_mapping_method", "raw_frame_mapping_ok", "mapping_pts_delta_ns"]].head(20))


## NvDCF Track Summary and Bounding-box Trajectories

Use `objects.csv` for trajectories: it contains both current tracker boxes and flags that distinguish detector observations from tracker predictions.


In [ ]:
if objects.empty:
    print("No object rows to plot.")
else:
    objects["cx"] = objects["track_x"] + objects["track_w"] / 2
    objects["cy"] = objects["track_y"] + objects["track_h"] / 2
    display(tracks.sort_values("total_rows", ascending=False).head(20))
    valid = objects.dropna(subset=["track_id", "frame_time_s", "cx", "cy"])
    top_ids = valid["track_id"].value_counts().head(8).index
    plot_objects = valid[valid["track_id"].isin(top_ids)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=plot_objects, x="frame_time_s", y="cx", hue="track_id", estimator=None, ax=axes[0])
    axes[0].set(xlabel="time (s)", ylabel="center x (px)", title="Longest tracks over time")
    sns.scatterplot(data=plot_objects, x="cx", y="cy", hue="track_id", size="detected_this_frame", sizes=(15, 50), ax=axes[1])
    axes[1].invert_yaxis(); axes[1].set_aspect("equal", adjustable="box")
    axes[1].set(xlabel="x (px)", ylabel="y (px)", title="Track paths; larger point = detector observation")
    plt.tight_layout()


## Pose Keypoints

Keypoints are loaded from the normalized `keypoints.csv` table. Names come from the selected model package; no class or pose indices are hard-coded here.


In [ ]:
def selected_pose_sidecar() -> Path:
    if RESULT_DIR != RUN_DIR:
        manifest = json.loads((RESULT_DIR / "offline_manifest.json").read_text())
        declared = Path(manifest["model_package"]["pose_sidecar"])
        if declared.exists():
            return declared
    snapshots = sorted((RUN_DIR / "config").glob("*.pose.json"))
    if len(snapshots) != 1:
        raise ValueError(f"expected one run-local pose schema, found {len(snapshots)}")
    return snapshots[0]

pose_schema = json.loads(selected_pose_sidecar().read_text())
print(f"model keypoints={pose_schema.get('keypoint_count')}")
if not keypoints.empty:
    display(keypoints.groupby("keypoint_name")["confidence"].describe().sort_values("mean", ascending=False))


In [ ]:
MIN_KEYPOINT_SCORE = float(pose_schema.get("keypoint_threshold", 0.5))
KEYPOINTS_TO_PLOT = list(keypoints["keypoint_name"].dropna().unique()[:6]) if not keypoints.empty else []

pose = keypoints[(keypoints["keypoint_name"].isin(KEYPOINTS_TO_PLOT)) & (keypoints["confidence"] >= MIN_KEYPOINT_SCORE)].copy()
if pose.empty:
    print("No keypoints pass the selected filter.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    sns.lineplot(data=pose, x="frame_time_s", y="x_px", hue="keypoint_name", estimator=None, ax=axes[0])
    sns.lineplot(data=pose, x="frame_time_s", y="y_px", hue="keypoint_name", estimator=None, ax=axes[1], legend=False)
    axes[0].set(title="Keypoint x", ylabel="x (px)")
    axes[1].set(title="Keypoint y", xlabel="time (s)", ylabel="y (px)")
    plt.tight_layout()


## Event-aligned Inference


In [ ]:
def event_locked_objects(event_type: str, pre_s=1.0, post_s=2.0) -> pd.DataFrame:
    anchors = events[(events["eventType"] == event_type) & events["event_time_s"].notna()]
    chunks = []
    for anchor_index, event in anchors.iterrows():
        t0 = float(event["event_time_s"])
        window = objects[objects["frame_time_s"].between(t0 - pre_s, t0 + post_s)].copy()
        window["anchor_index"] = anchor_index
        window["time_from_event_s"] = window["frame_time_s"] - t0
        chunks.append(window)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

EVENT_TYPE = "POKE_START"
locked = event_locked_objects(EVENT_TYPE)
print(f"{EVENT_TYPE} locked object rows: {len(locked):,}")
if not locked.empty:
    sns.scatterplot(data=locked, x="time_from_event_s", y="detector_confidence", hue="track_id", alpha=.6)
    plt.axvline(0, color="black", ls="--", lw=1)
    plt.title(f"Tracked objects around {EVENT_TYPE}")
    plt.tight_layout()


## Optional Exact-frame Raw Video Preview

This preview seeks by the authoritative raw-video frame index, not an approximate timestamp. It overlays tracker boxes and normalized keypoints from the selected result. Preview generation is disabled by default and requires a separate derived-results directory so the copied source run remains unchanged.


In [ ]:
RUN_PREVIEW = False
RAW_FRAME_INDEX_TO_PREVIEW = 12
PREVIEW_OUTPUT_DIR = None  # Set to a derived-results directory before enabling preview.
# RAW_FRAME_INDEX_TO_PREVIEW = int(frames.iloc[len(frames) // 2]["raw_frame_index"]) if len(frames) else 0
KEYPOINT_SCORE_THRESHOLD = float(pose_schema.get("keypoint_threshold", 0.5))

if RUN_PREVIEW:
    if PREVIEW_OUTPUT_DIR is None:
        raise ValueError("Set PREVIEW_OUTPUT_DIR outside the copied source run")
    preview_output_dir = Path(PREVIEW_OUTPUT_DIR).expanduser().resolve()
    preview_output_dir.mkdir(parents=True, exist_ok=True)
    frame_match = frames[frames["raw_frame_index"] == RAW_FRAME_INDEX_TO_PREVIEW]
    if frame_match.empty:
        raise ValueError(f"No ledger row for raw_frame_index={RAW_FRAME_INDEX_TO_PREVIEW}")
    frame_row = frame_match.iloc[0]
    video_path = RUN_DIR / "raw.mp4"
    local_index = int(frame_row.get("video_frame_index", frame_row["raw_frame_index"] - frames["raw_frame_index"].min()))
    out_png = preview_output_dir / f"preview_raw_{RAW_FRAME_INDEX_TO_PREVIEW:06d}.png"
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(video_path),
        "-vf", f"select=eq(n\\,{local_index})", "-vsync", "0", "-frames:v", "1", str(out_png),
    ], check=True)

    frame_objects = objects[objects["source_sequence_index"] == RAW_FRAME_INDEX_TO_PREVIEW]
    frame_points = keypoints[(keypoints["source_sequence_index"] == RAW_FRAME_INDEX_TO_PREVIEW) & (keypoints["confidence"] >= KEYPOINT_SCORE_THRESHOLD)]
    image = plt.imread(out_png)
    fig, ax = plt.subplots(figsize=(12, 8)); ax.imshow(image, cmap="gray")
    cmap = plt.get_cmap("tab10")
    for ordinal, (_, obj) in enumerate(frame_objects.iterrows()):
        color = cmap(ordinal % 10)
        ax.add_patch(Rectangle((obj["track_x"], obj["track_y"]), obj["track_w"], obj["track_h"], fill=False, edgecolor=color, lw=2))
        ax.text(obj["track_x"], max(0, obj["track_y"] - 5), f"{obj['class_label']} T{obj['track_id']}", color=color)
        points = frame_points[frame_points["observation_id"] == obj["observation_id"]]
        ax.scatter(points["x_px"], points["y_px"], color=[color], s=24)
    ax.set_title(f"raw_frame_index={RAW_FRAME_INDEX_TO_PREVIEW}; camera_frame_id={int(frame_row['camera_frame_id'])}")
    ax.set_xlim(0, image.shape[1]); ax.set_ylim(image.shape[0], 0); ax.axis("off")
    plt.tight_layout()
    display(frame_objects, frame_points)
else:
    print("Preview disabled. Set RUN_PREVIEW=True and choose RAW_FRAME_INDEX_TO_PREVIEW.")
